# 06 Visualization and Analysis

Notebook untuk membuat dan menampilkan visualisasi hasil CBR serta checklist validasi.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = DATA_DIR / 'results'
EVAL_DIR = DATA_DIR / 'eval'
PROCESSED_DIR = DATA_DIR / 'processed'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
cases_df = pd.read_csv(PROCESSED_DIR / 'cases.csv')
train_df = pd.read_csv(PROCESSED_DIR / 'case_base_train.csv')
test_df = pd.read_csv(EVAL_DIR / 'test_cases.csv')
retrieval_metrics = pd.read_csv(EVAL_DIR / 'retrieval_metrics.csv')
prediction_metrics = pd.read_csv(EVAL_DIR / 'prediction_metrics.csv')
confusion_matrix = pd.read_csv(EVAL_DIR / 'confusion_matrix.csv')


def annotate_bars(ax, bars, offset=0.02):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + offset, f'{height:g}', ha='center', va='bottom', fontsize=9)


def save_label_distribution():
    counts = cases_df['solution_label'].fillna('Kosong').value_counts().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    bars = ax.bar(counts.index, counts.values, color='#4f7cac')
    ax.set_title('Distribusi Label Solusi')
    ax.set_ylabel('Jumlah Kasus')
    ax.tick_params(axis='x', rotation=25)
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    annotate_bars(ax, bars, offset=0.2)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'label_distribution_bar.png', dpi=160)
    plt.close(fig)


def save_train_test_split():
    split_counts = pd.Series({'Case Base Train': len(train_df), 'Query Test': len(test_df)})
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    bars = ax.bar(split_counts.index, split_counts.values, color=['#4f7cac', '#c46d5e'])
    ax.set_title('Pembagian Data Train dan Test')
    ax.set_ylabel('Jumlah Kasus')
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    annotate_bars(ax, bars, offset=0.2)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'train_test_split_bar.png', dpi=160)
    plt.close(fig)


def save_word_count_distribution():
    word_counts = pd.to_numeric(cases_df['jumlah_kata'], errors='coerce').dropna()
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.hist(word_counts, bins=10, color='#6b8e6f', edgecolor='white')
    ax.set_title('Distribusi Jumlah Kata Dokumen')
    ax.set_xlabel('Jumlah Kata')
    ax.set_ylabel('Jumlah Kasus')
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'word_count_distribution.png', dpi=160)
    plt.close(fig)


def save_missing_fields():
    important_columns = ['no_perkara', 'tanggal_putusan', 'pengadilan', 'terdakwa', 'pasal', 'problem_text', 'solution_text', 'solution_label']
    missing_counts = cases_df[important_columns].isna().sum()
    fig, ax = plt.subplots(figsize=(9, 4.8))
    bars = ax.bar(missing_counts.index, missing_counts.values, color='#b36a5e')
    ax.set_title('Nilai Kosong pada Kolom Penting')
    ax.set_ylabel('Jumlah Kosong')
    ax.tick_params(axis='x', rotation=25)
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    annotate_bars(ax, bars, offset=0.05)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'missing_fields_bar.png', dpi=160)
    plt.close(fig)


def save_combined_metrics():
    combined = pd.concat([
        retrieval_metrics.assign(kategori='Retrieval'),
        prediction_metrics.assign(kategori='Prediction'),
    ], ignore_index=True)
    combined['label'] = combined['kategori'] + ': ' + combined['metric']
    combined['value'] = pd.to_numeric(combined['value'], errors='coerce')
    colors = combined['kategori'].map({'Retrieval': '#2f6f9f', 'Prediction': '#5b8c5a'}).fillna('#777777')

    fig, ax = plt.subplots(figsize=(11, 5.8))
    bars = ax.barh(combined['label'], combined['value'], color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Nilai')
    ax.set_title('Ringkasan Metrik Retrieval dan Prediksi')
    ax.grid(axis='x', linestyle='--', alpha=0.35)
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.015, bar.get_y() + bar.get_height() / 2, f'{width:.2f}', va='center', fontsize=9)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'combined_metrics_bar.png', dpi=160)
    plt.close(fig)


def save_confusion_matrix_heatmap():
    matrix = confusion_matrix.set_index(confusion_matrix.columns[0])
    fig, ax = plt.subplots(figsize=(8.5, 6.5))
    image = ax.imshow(matrix.values, cmap='Blues')
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_yticks(range(len(matrix.index)))
    ax.set_xticklabels(matrix.columns, rotation=35, ha='right')
    ax.set_yticklabels(matrix.index)
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Aktual')
    ax.set_title('Confusion Matrix Prediksi Solusi')
    max_value = matrix.values.max() if matrix.size else 0
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix.iat[i, j]
            color = 'white' if max_value and value > max_value / 2 else 'black'
            ax.text(j, i, str(value), ha='center', va='center', color=color)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'confusion_matrix_heatmap.png', dpi=160)
    plt.close(fig)


def save_validation_checklist():
    expected_images = [
        'label_distribution_bar.png',
        'train_test_split_bar.png',
        'word_count_distribution.png',
        'missing_fields_bar.png',
        'combined_metrics_bar.png',
        'confusion_matrix_heatmap.png',
    ]
    checks = [
        ('Dataset kasus tersedia', len(cases_df) > 0, f'{len(cases_df)} kasus'),
        ('Case base train tersedia', len(train_df) > 0, f'{len(train_df)} kasus'),
        ('Query test tersedia', len(test_df) > 0, f'{len(test_df)} kasus'),
        ('Train dan test konsisten', len(train_df) + len(test_df) == len(cases_df), f'{len(train_df)} + {len(test_df)} = {len(cases_df)}'),
        ('Label solusi lengkap', cases_df['solution_label'].notna().all(), f"{cases_df['solution_label'].isna().sum()} kosong"),
        ('Problem text lengkap', cases_df['problem_text'].notna().all(), f"{cases_df['problem_text'].isna().sum()} kosong"),
        ('Metrik retrieval tersedia', not retrieval_metrics.empty, f'{len(retrieval_metrics)} metrik'),
        ('Metrik prediksi tersedia', not prediction_metrics.empty, f'{len(prediction_metrics)} metrik'),
        ('Gambar visualisasi tersedia', all((RESULTS_DIR / name).exists() for name in expected_images), ', '.join(expected_images)),
    ]
    checklist = pd.DataFrame([
        {'aspek': aspect, 'status': 'OK' if passed else 'Perlu Cek', 'catatan': note}
        for aspect, passed, note in checks
    ])
    checklist.to_csv(EVAL_DIR / 'validation_checklist.csv', index=False)
    return checklist


save_label_distribution()
save_train_test_split()
save_word_count_distribution()
save_missing_fields()
save_combined_metrics()
save_confusion_matrix_heatmap()
validation_checklist = save_validation_checklist()
print('Artefak visualisasi dan checklist berhasil dibuat.')
display(validation_checklist)


## Distribusi Label Solusi

Grafik ini menunjukkan apakah label solusi seimbang atau tidak. Ketidakseimbangan label perlu dijelaskan karena dapat mempengaruhi metrik evaluasi.

In [ ]:
display(Image(filename=str(RESULTS_DIR / 'label_distribution_bar.png')))

## Pembagian Data Train dan Test

Train digunakan sebagai case base, sedangkan test digunakan sebagai query evaluasi. Pemisahan ini mencegah data leakage.

In [ ]:
display(Image(filename=str(RESULTS_DIR / 'train_test_split_bar.png')))

## Distribusi Jumlah Kata

Grafik ini menunjukkan variasi panjang putusan setelah ekstraksi teks dari PDF.

In [ ]:
display(Image(filename=str(RESULTS_DIR / 'word_count_distribution.png')))

## Missing Field

Grafik ini membantu melihat bagian metadata yang belum berhasil diekstrak secara sempurna dari PDF.

In [ ]:
display(Image(filename=str(RESULTS_DIR / 'missing_fields_bar.png')))

## Metrik Evaluasi

Grafik ini merangkum metrik retrieval dan prediksi solusi.

In [ ]:
display(Image(filename=str(RESULTS_DIR / 'combined_metrics_bar.png')))
print(pd.read_csv(EVAL_DIR / 'retrieval_metrics.csv'))
print(pd.read_csv(EVAL_DIR / 'prediction_metrics.csv'))

## Confusion Matrix

Confusion matrix menunjukkan label yang benar dan label hasil prediksi.

In [ ]:
display(Image(filename=str(RESULTS_DIR / 'confusion_matrix_heatmap.png')))

## Checklist Validasi

Checklist ini digunakan untuk memastikan file output utama sudah lengkap dan metodologi sudah menghindari leakage.

In [ ]:
pd.read_csv(EVAL_DIR / 'validation_checklist.csv')